The Look: E-commerce 
----------------------------------------------------------------
Preliminary analysis conducted on 20k sample for computational efficiency.

Dataset from bigquery-public-data: "thelook_ecommerce"
Objective: exploratory data analysis (EDA) to understand data structure, quality, and potential business insights.
Tools: BigQuery + Pandas


In [6]:
#1. IMPORT LIBRARIES & SETUP
  
from dotenv import load_dotenv    #to load .env file
import os                         #interact with Operating System
from google.cloud import bigquery #connect to BigQuery

import pandas as pd
import numpy as np

In [7]:
# 2. CONNECT TO BIGQUERY
load_dotenv()   #load file .env with google cloud project id

PROJECT_ID = os.getenv('PROJECT_ID')
print(f"Project ID {'[HIDDEN FOR SECURITY] - Loaded successfully' if PROJECT_ID else 'NONE FOUND (create .env file with PROJECT_ID)'}")

client = bigquery.Client(project=PROJECT_ID)
print("Connected to BigQuery successfully!")

Project ID [HIDDEN FOR SECURITY] - Loaded successfully
Connected to BigQuery successfully!


In [8]:
# 3. DATA SELECTION  (7 datasets)
queries = {
    "products": """
        SELECT * FROM `bigquery-public-data.thelook_ecommerce.products` 
        LIMIT 20000
    """,
    "users": """
        SELECT * EXCEPT(first_name, last_name, user_geom) 
        FROM `bigquery-public-data.thelook_ecommerce.users` 
        WHERE created_at <= CURRENT_TIMESTAMP()
        LIMIT 20000
    """,
    "orders_raw": """
        SELECT * FROM `bigquery-public-data.thelook_ecommerce.orders`
        WHERE created_at<= CURRENT_TIMESTAMP() AND shipped_at <= CURRENT_TIMESTAMP()
    """,
    "order_items": """
        SELECT * FROM `bigquery-public-data.thelook_ecommerce.order_items`
        WHERE created_at <= CURRENT_TIMESTAMP() AND shipped_at <= CURRENT_TIMESTAMP()
        LIMIT 20000
    """,
    "events": """
        SELECT * EXCEPT (ip_address, postal_code, uri)
        FROM `bigquery-public-data.thelook_ecommerce.events`
        WHERE created_at <= CURRENT_TIMESTAMP()
        LIMIT 100000
    """,
    "inventory_items": """
        SELECT * FROM `bigquery-public-data.thelook_ecommerce.inventory_items`
        WHERE created_at <= CURRENT_TIMESTAMP() AND sold_at <= CURRENT_TIMESTAMP()
        LIMIT 20000
    """,
    "distribution_centers": """
        SELECT * FROM `bigquery-public-data.thelook_ecommerce.distribution_centers`
    """}


In [9]:
# 4. LOAD PRODUCTS 
try:
    products = client.query(queries["products"]).to_dataframe()
    print(products.head(3))
    
except Exception:
    print("Something went wrong.")

      id     cost     category  \
0  13842  2.51875  Accessories   
1  13928  2.33835  Accessories   
2  14115  4.87956  Accessories   

                                                name brand  retail_price  \
0   Low Profile Dyed Cotton Twill Cap - Navy W39S55D    MG          6.25   
1  Low Profile Dyed Cotton Twill Cap - Putty W39S55D    MG          5.95   
2       Enzyme Regular Solid Army Caps-Black W35S45D    MG         10.99   

  department                               sku  distribution_center_id  
0      Women  EBD58B8A3F1D72F4206201DA62FB1204                       1  
1      Women  2EAC42424D12436BDD6A5B8A88480CC3                       1  
2      Women  EE364229B2791D1EF9355708EFF0BA34                       1  


In [10]:
# 4.1 PRODUCTS EDA 
print("PRODUCTS EXPLORATION")
print(products.info())
print("\n", products[["cost","retail_price"]].describe())

PRODUCTS EXPLORATION
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 9 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   id                      20000 non-null  Int64  
 1   cost                    20000 non-null  float64
 2   category                20000 non-null  object 
 3   name                    19998 non-null  object 
 4   brand                   19976 non-null  object 
 5   retail_price            20000 non-null  float64
 6   department              20000 non-null  object 
 7   sku                     20000 non-null  object 
 8   distribution_center_id  20000 non-null  Int64  
dtypes: Int64(2), float64(2), object(5)
memory usage: 1.4+ MB
None

                cost  retail_price
count  20000.000000  20000.000000
mean      28.519070     59.799238
std       30.492835     66.912015
min        0.008300      0.020000
25%       11.970157     25.000000
50%       20.04

In [11]:
print(f"\nTOP 10 Categories by Sum Retail Price:\n{(products.groupby('category')['retail_price'].agg(['mean', 'sum']).sort_values('sum', ascending=False).head(10))}")
print(f"\nWORST 10 Categories by Sum Retail Price:\n{(products.groupby('category')['retail_price'].agg(['mean', 'sum']).sort_values('sum', ascending=True).head(10))}")


TOP 10 Categories by Sum Retail Price:
                                     mean            sum
category                                                
Outerwear & Coats              151.556756  157922.139793
Jeans                           80.487072  105277.090262
Sweaters                        76.059350   88913.379993
Swim                            58.373059   75768.230050
Suits & Sport Coats            138.426539   68797.990015
Fashion Hoodies & Sweatshirts   56.599267   63334.580150
Sleep & Lounge                  48.797122   62557.910108
Dresses                         98.777984   60254.570129
Active                          51.039084   57418.970033
Intimates                       35.165904   57179.760116

WORST 10 Categories by Sum Retail Price:
                           mean           sum
category                                     
Clothing Sets         85.350526   1621.659988
Jumpsuits & Rompers   76.494167   4589.650009
Socks & Hosiery       17.064022   9504.659988
Legg

In [12]:
# 5. LOAD USERS
try:
    users = client.query(queries["users"]).to_dataframe()
    print(users.head())
    
except Exception:
    print("Something went wrong.")

      id                      email  age gender state  \
0   1027  aliciaparsons@example.org   20      F  Acre   
1  61389       marytran@example.org   59      F  Acre   
2   3576  kimberlymoran@example.com   40      F  Acre   
3  39749   deannageorge@example.com   60      F  Acre   
4  12301   patriciahall@example.com   20      F  Acre   

                    street_address postal_code  city country  latitude  \
0          041 Wood Lodge Apt. 613   69980-000  null  Brasil -8.065346   
1    6470 Lindsay Highway Apt. 767   69980-000  null  Brasil -8.065346   
2  27232 Zimmerman Spring Apt. 484   69980-000  null  Brasil -8.065346   
3                3708 Anthony View   69980-000  null  Brasil -8.065346   
4      6582 Burnett Flats Apt. 826   69980-000  null  Brasil -8.065346   

   longitude traffic_source                created_at  
0 -72.870949        Display 2025-10-01 03:41:00+00:00  
1 -72.870949        Organic 2020-03-07 05:13:00+00:00  
2 -72.870949        Organic 2024-12-31 06:16

In [13]:
# 5.1 USERS EDA
print("USERS EXPLORATION")
print(users.info())

USERS EXPLORATION
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype              
---  ------          --------------  -----              
 0   id              20000 non-null  Int64              
 1   email           20000 non-null  object             
 2   age             20000 non-null  Int64              
 3   gender          20000 non-null  object             
 4   state           20000 non-null  object             
 5   street_address  20000 non-null  object             
 6   postal_code     20000 non-null  object             
 7   city            20000 non-null  object             
 8   country         20000 non-null  object             
 9   latitude        20000 non-null  float64            
 10  longitude       20000 non-null  float64            
 11  traffic_source  20000 non-null  object             
 12  created_at      20000 non-null  datetime64[us, UTC]
dtypes: Int64(2), 

In [14]:
print("Age users \n", users["age"].describe())

Age users 
 count      20000.0
mean      41.03335
std      17.105332
min           12.0
25%           26.0
50%           41.0
75%           56.0
max           70.0
Name: age, dtype: Float64


In [15]:
print("\n", users[['id','email','city']].nunique())
email_duplicate = users[['email']].duplicated(keep=False)
email_duplicated = users.loc[email_duplicate,["id","age", "created_at", "email", "city", "country"]].sort_values('email')
print("\n",email_duplicated)  #We have different users with the same email but age discrepancies and location.


 id       20000
email    19092
city      2033
dtype: int64

           id  age                created_at                          email  \
10013  46694   59 2022-06-05 07:41:00+00:00      adamhernandez@example.com   
6641   29195   35 2023-02-04 12:58:00+00:00      adamhernandez@example.com   
19249  64466   19 2022-10-12 05:26:00+00:00        adamjohnson@example.org   
11670  11120   35 2020-06-25 08:27:00+00:00        adamjohnson@example.org   
14457  24523   14 2022-07-10 00:21:00+00:00          adamsmith@example.net   
...      ...  ...                       ...                            ...   
8168   82086   20 2022-08-25 16:03:00+00:00        zacharydiaz@example.org   
12208  95155   38 2024-02-18 13:24:00+00:00  zacharyrichardson@example.com   
11935  17152   27 2022-04-08 17:40:00+00:00  zacharyrichardson@example.com   
1548    7114   55 2024-03-07 10:17:00+00:00     zacharyshannon@example.net   
5622   69705   66 2024-03-22 02:53:00+00:00     zacharyshannon@example.net   

 

In [16]:
print("\n", users['gender'].value_counts(normalize=True))
print("\n", users['country'].value_counts())
print("\n Account creation date:", users['created_at'].agg(['min', 'max']))


 gender
F    0.5011
M    0.4989
Name: proportion, dtype: float64

 country
United States     5526
Brasil            3672
China             3404
Spain             3319
Germany           1353
France            1119
South Korea       1022
Japan              243
Belgium            146
Poland              82
United Kingdom      62
Australia           42
Colombia            10
Name: count, dtype: int64

 Account creation date: min          2019-01-02 03:11:00+00:00
max   2026-03-19 19:29:05.234921+00:00
Name: created_at, dtype: datetime64[ns, UTC]


In [17]:
# 6. LOAD ORDERS 
try:
    orders_raw = client.query(queries["orders_raw"]).to_dataframe()
    print(orders_raw.head())
    
except Exception:
    print("Something went wrong.")

   order_id  user_id    status gender                created_at returned_at  \
0         7        4  Complete      F 2025-09-18 22:35:08+00:00         NaT   
1        19       15  Complete      F 2023-12-26 14:26:48+00:00         NaT   
2        24       17  Complete      F 2024-05-13 16:53:10+00:00         NaT   
3        25       18  Complete      F 2025-09-01 07:22:34+00:00         NaT   
4        32       25  Complete      F 2024-01-16 21:56:27+00:00         NaT   

                 shipped_at              delivered_at  num_of_item  
0 2025-09-20 19:55:08+00:00 2025-09-23 03:11:08+00:00            1  
1 2023-12-29 12:51:48+00:00 2024-01-03 11:46:48+00:00            1  
2 2024-05-14 20:20:10+00:00 2024-05-15 01:38:10+00:00            1  
3 2025-09-04 03:15:34+00:00 2025-09-04 16:58:34+00:00            2  
4 2024-01-19 14:19:27+00:00 2024-01-20 00:46:27+00:00            1  


In [18]:
# 6.1 ORDERS EDA
print("ORDERS EXPLORATION")
print(orders_raw.info())
print("\n", orders_raw["num_of_item"].describe())

ORDERS EXPLORATION
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 80393 entries, 0 to 80392
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype              
---  ------        --------------  -----              
 0   order_id      80393 non-null  Int64              
 1   user_id       80393 non-null  Int64              
 2   status        80393 non-null  object             
 3   gender        80393 non-null  object             
 4   created_at    80393 non-null  datetime64[us, UTC]
 5   returned_at   12554 non-null  datetime64[us, UTC]
 6   shipped_at    80393 non-null  datetime64[us, UTC]
 7   delivered_at  43474 non-null  datetime64[us, UTC]
 8   num_of_item   80393 non-null  Int64              
dtypes: Int64(3), datetime64[us, UTC](4), object(2)
memory usage: 5.8+ MB
None

 count     80393.0
mean     1.449591
std      0.803592
min           1.0
25%           1.0
50%           1.0
75%           2.0
max           4.0
Name: num_of_item, dtype: Float64


In [19]:
print("\n", orders_raw['gender'].value_counts(normalize=True))
print("\n", orders_raw['status'].value_counts(normalize=True))
print("\nRange", orders_raw[['created_at','shipped_at','delivered_at']].agg(['min', 'max']))


 gender
M    0.500131
F    0.499869
Name: proportion, dtype: float64

 status
Shipped     0.459232
Complete    0.384611
Returned    0.156158
Name: proportion, dtype: float64

Range                           created_at                       shipped_at  \
min        2019-01-05 08:30:43+00:00        2019-01-05 09:12:43+00:00   
max 2026-03-20 00:45:52.288734+00:00 2026-03-20 14:16:12.529082+00:00   

                        delivered_at  
min        2019-01-15 10:00:12+00:00  
max 2026-03-25 13:00:59.251093+00:00  


In [20]:
orders_raw['shipping_duration'] = orders_raw['delivered_at'] - orders_raw['shipped_at']
shipping_duration = orders_raw['shipping_duration'].dropna()
print("Orders delivered (%):", round(orders_raw['shipping_duration'].notna().mean() * 100, 2))
print("\nShipping duration (just delivered orders):\n", shipping_duration.describe())


Orders delivered (%): 54.08

Shipping duration (just delivered orders):
 count                     43474
mean     2 days 12:05:55.028752
std      1 days 10:43:28.724330
min             0 days 00:00:00
25%             1 days 05:54:00
50%             2 days 12:06:30
75%             3 days 18:23:00
max             4 days 23:59:00
Name: shipping_duration, dtype: object


In [21]:
in_transit = (orders_raw['shipped_at'].notnull() & orders_raw['delivered_at'].isnull())
transit_orders = orders_raw.loc[in_transit,['num_of_item', 'status', 'created_at', 'shipped_at','delivered_at']].sort_values('shipped_at') 

print("Shipped orders but not yet delivered:\n",transit_orders.agg({'shipped_at': ['count','min','max']}))
print("\n Items in transit (as % of total items):", round(transit_orders['num_of_item'].sum()/(orders_raw['num_of_item'].sum())*100,2))


Shipped orders but not yet delivered:
                       shipped_at
count                      36919
min    2019-01-05 09:12:43+00:00
max    2026-03-20 14:14:48+00:00

 Items in transit (as % of total items): 45.9


In [22]:
# 7. LOAD ORDERS-ITEMS
try:
    order_items = client.query(queries["order_items"]).to_dataframe()
    print(order_items.head())
    
except Exception:
    print("Something went wrong.")

       id  order_id  user_id  product_id  inventory_item_id    status  \
0  138842     95786    76689       14235             374962   Shipped   
1   56687     39245    31552       14159             153192  Complete   
2  178300    123057    98380       14159             481724   Shipped   
3  136954     94506    75653       28700             369826  Complete   
4   54143     37470    30077       28700             146269  Returned   

                 created_at                shipped_at  \
0 2026-01-19 17:30:13+00:00 2026-01-19 13:16:33+00:00   
1 2024-12-03 08:06:35+00:00 2024-12-04 15:03:01+00:00   
2 2025-07-13 17:58:18+00:00 2025-07-13 19:22:40+00:00   
3 2020-05-04 13:52:17+00:00 2020-05-04 00:00:16+00:00   
4 2022-11-19 13:34:48+00:00 2022-11-20 23:00:43+00:00   

               delivered_at               returned_at  sale_price  
0                       NaT                       NaT        0.02  
1 2024-12-06 13:01:01+00:00                       NaT        0.49  
2             

In [23]:
# 7.1 ORDERS-ITEMS EDA
print("ORDER-ITEMS EXPLORATION")
print(order_items.info())
print("\n", order_items['sale_price'].describe())

ORDER-ITEMS EXPLORATION
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype              
---  ------             --------------  -----              
 0   id                 20000 non-null  Int64              
 1   order_id           20000 non-null  Int64              
 2   user_id            20000 non-null  Int64              
 3   product_id         20000 non-null  Int64              
 4   inventory_item_id  20000 non-null  Int64              
 5   status             20000 non-null  object             
 6   created_at         20000 non-null  datetime64[us, UTC]
 7   shipped_at         20000 non-null  datetime64[us, UTC]
 8   delivered_at       10887 non-null  datetime64[us, UTC]
 9   returned_at        3129 non-null   datetime64[us, UTC]
 10  sale_price         20000 non-null  float64            
dtypes: Int64(5), datetime64[us, UTC](4), float64(1), object(1)
memory usage: 1.8+ MB
N

In [24]:
print(order_items.loc[order_items['sale_price']<0.5]) # to check if there is any discrepancy in the selling prices and the products


       id  order_id  user_id  product_id  inventory_item_id    status  \
0  138842     95786    76689       14235             374962   Shipped   
1   56687     39245    31552       14159             153192  Complete   
2  178300    123057    98380       14159             481724   Shipped   

                 created_at                shipped_at  \
0 2026-01-19 17:30:13+00:00 2026-01-19 13:16:33+00:00   
1 2024-12-03 08:06:35+00:00 2024-12-04 15:03:01+00:00   
2 2025-07-13 17:58:18+00:00 2025-07-13 19:22:40+00:00   

               delivered_at returned_at  sale_price  
0                       NaT         NaT        0.02  
1 2024-12-06 13:01:01+00:00         NaT        0.49  
2                       NaT         NaT        0.49  


In [25]:
print(products.loc[(products['id'] == 14159) | (products['id'] == 14235)])

         id     cost     category  \
1401  14235  0.00830  Accessories   
2300  14159  0.17738  Accessories   

                                                   name        brand  \
1401         Indestructable Aluminum Aluma Wallet - RED      marshal   
2300  Set of 2 - Replacement Insert For Checkbook Wa...  Made in USA   

      retail_price department                               sku  \
1401          0.02      Women  8425BC94A44E3D1BB3C8C026B2702C00   
2300          0.49      Women  C92B32FBC94E2DFF3E5516401D9BB463   

      distribution_center_id  
1401                       1  
2300                       1  


In [26]:
print("\n", order_items['status'].value_counts(normalize=True))
print("\nRange", order_items[['created_at','shipped_at','delivered_at']].agg(['min', 'max']))


 status
Shipped     0.45565
Complete    0.38790
Returned    0.15645
Name: proportion, dtype: float64

Range                           created_at                       shipped_at  \
min        2019-01-05 05:07:04+00:00        2019-01-05 09:12:43+00:00   
max 2026-03-20 13:59:28.958575+00:00 2026-03-20 14:11:24.629931+00:00   

                        delivered_at  
min        2019-01-18 07:09:53+00:00  
max 2026-03-25 08:10:00.612291+00:00  


In [27]:
# 8. LOAD INVENTORY-ITEMS
try:
    inventory_items = client.query(queries["inventory_items"]).to_dataframe()
    print(inventory_items.sample(3))
    
except Exception:
    print("Something went wrong.")

           id  product_id                created_at                   sold_at  \
17669  117850       18350 2024-07-04 15:16:24+00:00 2024-08-26 04:24:24+00:00   
7275   360779       13777 2025-11-26 16:07:01+00:00 2026-01-09 19:04:01+00:00   
1290   233180       28444 2026-02-19 17:42:07+00:00 2026-02-24 01:54:07+00:00   

            cost product_category  \
17669   5.399680           Active   
7275   36.190521      Accessories   
1290    9.021390      Accessories   

                                            product_name product_brand  \
17669  UnderArmour Men's Grometric Performance Crew S...  Under Armour   
7275            Ray-Ban RB4141 Round Wayfarer Sunglasses       Ray-Ban   
1290                 Calvin Klein Men's Leather Bookfold  Calvin Klein   

       product_retail_price product_department  \
17669             12.980000                Men   
7275              91.160004              Women   
1290              24.990000                Men   

                            

In [28]:
# 8.1 INVENTORY-ITEMS EDA
print("INVENTORY-ITEMS EXPLORATION")
print(inventory_items.info())

INVENTORY-ITEMS EXPLORATION
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 12 columns):
 #   Column                          Non-Null Count  Dtype              
---  ------                          --------------  -----              
 0   id                              20000 non-null  Int64              
 1   product_id                      20000 non-null  Int64              
 2   created_at                      20000 non-null  datetime64[us, UTC]
 3   sold_at                         20000 non-null  datetime64[us, UTC]
 4   cost                            20000 non-null  float64            
 5   product_category                20000 non-null  object             
 6   product_name                    20000 non-null  object             
 7   product_brand                   20000 non-null  object             
 8   product_retail_price            20000 non-null  float64            
 9   product_department              20000 non-null  object 

In [29]:
print(inventory_items['product_distribution_center_id'].value_counts(normalize=True))
print("\n",inventory_items['product_category'].value_counts(normalize=True))
print("\n",inventory_items['product_brand'].value_counts(normalize=True).head(3))
print("\nData range", inventory_items[['created_at','sold_at']].agg(['min', 'max']))

product_distribution_center_id
2     0.18295
1     0.15985
3     0.11015
9      0.1021
6      0.0932
4     0.09155
7       0.083
8     0.07645
10    0.06065
5      0.0401
Name: proportion, dtype: Float64

 product_category
Accessories          0.48885
Active               0.45475
Blazers & Jackets    0.05640
Name: proportion, dtype: float64

 product_brand
Allegra K    0.04590
Champion     0.03515
adidas       0.03165
Name: proportion, dtype: float64

Data range                           created_at                          sold_at
min        2018-12-24 13:47:51+00:00        2019-01-15 12:01:51+00:00
max 2026-03-19 18:13:34.789799+00:00 2026-03-20 14:11:36.530213+00:00


In [30]:
inventory_items['margin'] = inventory_items['product_retail_price'] - inventory_items['cost']
inventory_items['margin_pct'] = inventory_items['margin'] / inventory_items['product_retail_price']
print(inventory_items[['cost','product_retail_price','margin','margin_pct']].describe())   #all products have more than 50% margin

               cost  product_retail_price        margin    margin_pct
count  20000.000000          20000.000000  20000.000000  20000.000000
mean      18.686745             45.595779     26.909034      0.591025
std       26.616442             64.352243     37.895759      0.031041
min        0.008300              0.020000      0.011700      0.530000
25%        6.520650             15.990000      9.623580      0.566000
50%       11.186400             26.990000     15.978080      0.591000
75%       21.600000             51.000000     30.184500      0.616000
max      403.641002            903.000000    532.769998      0.669000


In [31]:
print(inventory_items.groupby('product_department')['margin_pct'].mean().sort_values(ascending=False))

product_department
Women    0.593487
Men      0.588974
Name: margin_pct, dtype: float64


In [32]:
inventory_items['days_to_sell'] = (inventory_items['sold_at'] - inventory_items['created_at']).dt.days
print(inventory_items['days_to_sell'].describe())

count    20000.000000
mean        29.400300
std         17.316257
min          0.000000
25%         14.000000
50%         30.000000
75%         44.000000
max         59.000000
Name: days_to_sell, dtype: float64


In [33]:
print(inventory_items.groupby('product_distribution_center_id')[['days_to_sell','margin_pct']].mean())

                                days_to_sell  margin_pct
product_distribution_center_id                          
1                                  29.503284    0.590099
2                                  29.398743    0.590335
3                                  29.405356    0.585647
4                                  29.395958    0.587803
5                                  28.839152    0.594939
6                                  29.123927    0.589689
7                                  29.272289    0.588627
8                                  29.456508    0.593877
9                                  29.735064    0.600950
10                                 29.467436    0.592619


In [34]:
# 9. LOAD EVENTS
try:
    events = client.query(queries["events"]).to_dataframe()
    print(events.head())
    
except Exception:
    print("Something went wrong.")

        id  user_id  sequence_number                            session_id  \
0  1684299     <NA>                3  c7a04292-7d4c-4c92-a66d-d17c2c6c19da   
1  1998177     <NA>                3  3c8c5046-2e6b-4d9c-9c71-408f1cf4fbac   
2  1541799     <NA>                3  610b8890-1395-4558-a730-21e53e44bd3d   
3  1645772     <NA>                3  e85348bb-c4cd-4e1d-a92b-e0bb034b5187   
4  1485580     <NA>                3  60169476-0d01-4469-9d74-eb945192390e   

                 created_at       city      state  browser traffic_source  \
0 2023-11-08 04:43:00+00:00  Loda City      Akita   Chrome        Adwords   
1 2022-10-13 10:52:00+00:00  São Paulo  São Paulo   Chrome          Email   
2 2024-07-31 04:50:00+00:00  São Paulo  São Paulo   Chrome          Email   
3 2022-06-02 04:21:00+00:00  São Paulo  São Paulo   Chrome        Organic   
4 2020-10-08 04:06:00+00:00  São Paulo  São Paulo  Firefox        YouTube   

  event_type  
0     cancel  
1     cancel  
2     cancel  
3     ca

In [35]:
# 9.1 EVENTS EDA
print("EVENTS EXPLORATION")
print(events.info())

EVENTS EXPLORATION
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 10 columns):
 #   Column           Non-Null Count   Dtype              
---  ------           --------------   -----              
 0   id               100000 non-null  Int64              
 1   user_id          16573 non-null   Int64              
 2   sequence_number  100000 non-null  Int64              
 3   session_id       100000 non-null  object             
 4   created_at       100000 non-null  datetime64[us, UTC]
 5   city             100000 non-null  object             
 6   state            100000 non-null  object             
 7   browser          100000 non-null  object             
 8   traffic_source   100000 non-null  object             
 9   event_type       100000 non-null  object             
dtypes: Int64(3), datetime64[us, UTC](1), object(6)
memory usage: 7.9+ MB
None


In [36]:
print(events['state'].unique())   #For info about location user, better to merge with users table. 

['Akita' 'São Paulo' 'Iwate' 'Aomori' 'Brussels' 'Grand Est' 'Beijing'
 'New York' 'Inner Mongolia Autonomous Region' 'Massachusetts'
 'País Vasco' 'Berlin' 'Remote territory' 'Tokyo' 'Occitanie' 'Liaoning'
 'Andalucía' 'Sachsen' 'Seoul' 'Comunidad Valenciana' 'Jilin'
 "Provence-Alpes-Côte d'Azur" 'Castilla-La Mancha' 'Wallonia' 'Normandie'
 'Brandenburg' 'Auvergne-Rhône-Alpes' 'Flanders' 'Heilongjiang' 'Galicia'
 'Pennsylvania' 'Cataluña' 'Nouvelle-Aquitaine' 'Mecklenburg-Vorpommern'
 'Centre-Val de Loire' 'Delaware' 'Gangwon-do' 'Corse' 'Shanghai'
 'District of Columbia' 'Virginia' 'New South Wales' 'Rio de Janeiro'
 'Maryland' 'Kanagawa' 'Hauts-de-France' 'Bourgogne-Franche-Comté'
 'Jiangsu' 'Hamburg' 'Niedersachsen' 'Schleswig-Holstein' 'Aragón'
 'Bretagne' 'Anhui' 'Shandong' 'Chiba' 'La Rioja' 'West Virginia'
 'North Carolina' 'Comunidad de Madrid' 'Bremen' 'Rhode Island'
 'South Carolina' 'Australian Capital Territory' 'Espírito Santo' 'Shanxi'
 'Región de Murcia' 'Tianjin' 'Geor

In [37]:
print(events.agg({'user_id': ['count','size','nunique'], 'session_id': ['count','size','nunique']}))
print("\n NaN user_id %:",events['user_id'].isna().mean() * 100)

         user_id  session_id
count      16573      100000
size      100000      100000
nunique     9445       94098

 NaN user_id %: 83.42699999999999


In [38]:
logged_users = events['user_id'].notna()
print(pd.crosstab( events['event_type'],logged_users, normalize='index')) #all cancel is from non logged users (NAN)
print("\n",pd.crosstab(events['traffic_source'],logged_users, normalize='index'))

user_id        False     True 
event_type                    
cancel      1.000000  0.000000
cart        0.425645  0.574355

 user_id            False     True 
traffic_source                    
Adwords         0.835504  0.164496
Email           0.834341  0.165659
Facebook        0.833601  0.166399
Organic         0.832334  0.167666
YouTube         0.831885  0.168115


In [39]:
# 10. LOAD DISTRIBUTION CENTERS
try:
    distribution_centers = client.query(queries["distribution_centers"]).to_dataframe()
    print(distribution_centers.info())
    
except Exception:
    print("Something went wrong.")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   id                        10 non-null     Int64  
 1   name                      10 non-null     object 
 2   latitude                  10 non-null     float64
 3   longitude                 10 non-null     float64
 4   distribution_center_geom  10 non-null     object 
dtypes: Int64(1), float64(2), object(2)
memory usage: 542.0+ bytes
None


In [40]:
# 10.1 DISTRIBUTION CENTERS EDA
print(distribution_centers.head(10).sort_values(by='id'))

   id                                         name  latitude  longitude  \
5   1                                   Memphis TN   35.1174   -89.9711   
4   2                                   Chicago IL   41.8369   -87.6847   
7   3                                   Houston TX   29.7604   -95.3698   
0   4                               Los Angeles CA   34.0500  -118.2500   
3   5                               New Orleans LA   29.9500   -90.0667   
6   6  Port Authority of New York/New Jersey NY/NJ   40.6340   -73.7834   
8   7                              Philadelphia PA   39.9500   -75.1667   
1   8                                    Mobile AL   30.6944   -88.0431   
2   9                                Charleston SC   32.7833   -79.9333   
9  10                                  Savannah GA   32.0167   -81.1167   

  distribution_center_geom  
5  POINT(-89.9711 35.1174)  
4  POINT(-87.6847 41.8369)  
7  POINT(-95.3698 29.7604)  
0     POINT(-118.25 34.05)  
3    POINT(-90.0667 29.95)  
